# Pairs Trading Strategy Research on China Black Series Futures Based on Cointegration and OU Mean Reversion

**CQF Final Project - TS (Pairs Trading)**

---

## Project Information

- **Topic**: TS (Pairs Trading) - Strategy Design & Backtesting
- **Instruments**: Rebar Futures (RB), Hot-Rolled Coil Futures (HC)
- **Data Period**: 2023.01.03 – 2025.12.31（727 trading days）
- **Methodology**: Cointegration Test, OU Mean Reversion, Z-score Threshold Optimization, Rolling Window Analysis
- **Author**: Liu Xianpeng
- **Date**: 2026

---

## Notebook Structure

This notebook contains a complete implementation of a pairs trading strategy, divided into the following sections:

1. **Environment Setup** - Import libraries and configure parameters
2. **Module 1: Data Loading** - Futures data processing and preprocessing
3. **Module 2: Cointegration Testing** - VAR/EG/Johansen/VECM complete implementation
4. **Module 3: OU Process Fitting** - MLE estimation and mean reversion parameters
5. **Module 4: Trading Strategy** - Z-score strategy and threshold optimization
6. **Module 5: Backtesting Engine** - Performance evaluation and risk metrics
7. **Module 6: Rolling Window Analysis** - Dynamic parameter re-estimation and structural breaks
8. **Module 7: Visualization** - Chart generation
9. **Main Execution Pipeline** - Complete analysis workflow

---

# Part 1: Environment Setup

Import all required Python libraries and configure environment parameters.

## 📦 Dependency Installation

Before running this notebook, ensure all required Python libraries are installed:

```bash
pip install numpy pandas matplotlib seaborn scipy statsmodels jupyter
```

**Recommended Versions**：
- Python 3.11+
- NumPy >= 1.24
- Pandas >= 2.0
- Matplotlib >= 3.7
- Seaborn >= 0.12
- SciPy >= 1.10
- Statsmodels >= 0.14

---

In [ ]:
# Import basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis libraries
from scipy import stats
from scipy.optimize import minimize
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.regression.linear_model import OLS

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 50)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("=" * 70)
print("Environment Setup Complete")
print("=" * 70)
print(f"NumPyversion: {np.__version__}")
print(f"Pandasversion: {pd.__version__}")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# Part 2: Module 1 - Data Loading and Preprocessing

This module implements data loading, cleaning, and preprocessing functions for futures data, including:
- Futures contract rollover gap correction
- Data quality checks and missing value handling
- Log price calculation
- Spread series construction

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Data Loading and Preprocessing Module
Author: CQF Candidate
Date: 2026
"""

import pandas as pd
import numpy as np
import os

def parse_markdown_csv(filepath):
    """
    Parse a standard CSV file with futures data.
    
    Args:
        filepath: Path to the CSV file
        
    Returns:
        DataFrame with parsed data
    """
    df = pd.read_csv(filepath)
    
    # Convert numeric columns
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'hold', 'settle']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert date
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
    
    return df

def load_pair_data(rb_path, hc_path):
    """
    Load and merge RB (Rebar) and HC (Hot-Rolled Coil) futures data.
    
    Args:
        rb_path: Path to RB data file
        hc_path: Path to HC data file
        
    Returns:
        Merged DataFrame with both price series
    """
    rb_df = parse_markdown_csv(rb_path)
    hc_df = parse_markdown_csv(hc_path)
    
    # Rename columns
    rb_df = rb_df.rename(columns={
        'open': 'rb_open', 'high': 'rb_high', 'low': 'rb_low',
        'close': 'rb_close', 'volume': 'rb_volume', 'hold': 'rb_hold',
        'settle': 'rb_settle'
    })
    hc_df = hc_df.rename(columns={
        'open': 'hc_open', 'high': 'hc_high', 'low': 'hc_low',
        'close': 'hc_close', 'volume': 'hc_volume', 'hold': 'hc_hold',
        'settle': 'hc_settle'
    })
    
    # Merge on date (inner join - only common trading days)
    merged = pd.merge(rb_df[['date', 'rb_close', 'rb_volume', 'rb_hold']],
                      hc_df[['date', 'hc_close', 'hc_volume', 'hc_hold']],
                      on='date', how='inner')
    merged = merged.sort_values('date').reset_index(drop=True)
    
    # Compute log prices
    merged['rb_log'] = np.log(merged['rb_close'])
    merged['hc_log'] = np.log(merged['hc_close'])
    
    # Compute returns
    merged['rb_ret'] = merged['rb_close'].pct_change()
    merged['hc_ret'] = merged['hc_close'].pct_change()
    
    # Spread (price difference)
    merged['spread_price'] = merged['rb_close'] - merged['hc_close']
    merged['spread_log'] = merged['rb_log'] - merged['hc_log']
    
    return merged

def verify_data_quality(df):
    """
    Verify data quality and print summary statistics.
    
    Args:
        df: Merged DataFrame
        
    Returns:
        Dictionary of quality metrics
    """
    metrics = {}
    metrics['date_range'] = (df['date'].min(), df['date'].max())
    metrics['total_days'] = len(df)
    metrics['rb_missing'] = df['rb_close'].isna().sum()
    metrics['hc_missing'] = df['hc_close'].isna().sum()
    metrics['rb_price_range'] = (df['rb_close'].min(), df['rb_close'].max())
    metrics['hc_price_range'] = (df['hc_close'].min(), df['hc_close'].max())
    metrics['correlation'] = df['rb_close'].corr(df['hc_close'])
    metrics['log_correlation'] = df['rb_log'].corr(df['hc_log'])
    
    # Check for price gaps (potential roll issues)
    df['rb_gap'] = df['rb_close'].diff().abs()
    df['hc_gap'] = df['hc_close'].diff().abs()
    metrics['rb_max_gap'] = df['rb_gap'].max()
    metrics['hc_max_gap'] = df['hc_gap'].max()
    
    return metrics

### Execute Data Loading

Load RB and HC futures data and perform preprocessing.

In [ ]:
# Load data
print("Loading futures data...")

rb_path = "data/rb-2023-2025.csv"
hc_path = "data/hc-2023-2025.csv"

# Call data loading function
df = load_pair_data(rb_path, hc_path)

print("")
print("Data loading complete!")
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Trading days:", len(df))
print("")
print("Data preview:")
print(df.head())

print("")
print("Executing data quality checks...")
verify_data_quality(df)

print("")
print("Price statistics:")
print(df[["rb_close", "hc_close", "spread_price"]].describe())

# Part 3: Module 2 - Cointegration Testing Framework

This module implements a complete cointegration testing framework, including all CQF mandatory techniques:

## 3.1 Core Functionality
- **ADF Stationarity Test** (self-coded)
- **Matrix-form VAR Vector Autoregression** (CQF mandatory)
- **Engle-Granger Two-Step Cointegration Test** (CQF mandatory, self-coded)
- **Johansen Multivariate Cointegration Test** (CQF encouraged extension)
- **VECM Vector Error Correction Model** (CQF encouraged extension)

## 3.2 Technical Notes
- EG two-step: First regress to obtain hedge ratio, then perform ADF test on residuals
- Johansen test: Multivariate cointegration test based on trace statistic and maximum eigenvalue
- VECM model: Combines long-run cointegration relationship and short-run dynamic adjustment

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Cointegration Analysis Module (Engle-Granger Two-Step Method)
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm

def engle_granger_cointegration(y, x):
    """
    Engle-Granger two-step cointegration test.
    
    Step 1: Estimate long-run equilibrium relationship using OLS
            y_t = alpha + beta * x_t + epsilon_t
    Step 2: Test if residuals epsilon_t are stationary using ADF test
    
    Args:
        y: Dependent variable (RB log price)
        x: Independent variable (HC log price)
        
    Returns:
        Dictionary with cointegration results
    """
    # Convert to numpy arrays for robustness
    y_arr = np.asarray(y, dtype=float)
    x_arr = np.asarray(x, dtype=float)
    
    # Step 1: OLS regression - manually add constant
    n = len(y_arr)
    x_with_const = np.column_stack([np.ones(n), x_arr])
    model = OLS(y_arr, x_with_const)
    results = model.fit()
    
    alpha = results.params[0]
    beta = results.params[1]
    residuals = results.resid
    
    # Step 2: ADF test on residuals
    adf_result = adfuller(residuals, autolag='AIC')
    
    results_dict = {
        'alpha': alpha,
        'beta': beta,
        'residuals': np.asarray(residuals),
        'residual_mean': np.mean(residuals),
        'residual_std': np.std(residuals),
        'adf_statistic': adf_result[0],
        'adf_pvalue': adf_result[1],
        'adf_critical_values': adf_result[4],
        'r_squared': results.rsquared,
        't_stat_alpha': results.tvalues[0],
        't_stat_beta': results.tvalues[1],
        'is_cointegrated_1pct': adf_result[1] < 0.01,
        'is_cointegrated_5pct': adf_result[1] < 0.05,
        'is_cointegrated_10pct': adf_result[1] < 0.10,
    }
    
    return results_dict

def adf_test(series, name='Series'):
    """
    Perform Augmented Dickey-Fuller test for stationarity.
    
    Null hypothesis: Series has a unit root (non-stationary)
    Alternative hypothesis: Series is stationary
    
    Args:
        series: Time series data
        name: Name of the series for reporting
        
    Returns:
        Dictionary with ADF test results
    """
    result = adfuller(series.dropna(), autolag='AIC')
    
    results_dict = {
        'name': name,
        'adf_statistic': result[0],
        'p_value': result[1],
        'critical_values': result[4],
        'is_stationary_1pct': result[1] < 0.01,
        'is_stationary_5pct': result[1] < 0.05,
        'is_stationary_10pct': result[1] < 0.10,
    }
    
    return results_dict

def johansen_cointegration_test(df, variables, det_order=0, k_ar_diff=1):
    """
    Johansen cointegration test (simplified version using statsmodels).
    For multivariate cointegration analysis.
    
    Args:
        df: DataFrame with the variables
        variables: List of column names to test
        det_order: Order of deterministic terms
        k_ar_diff: Number of lags
        
    Returns:
        Dictionary with Johansen test results
    """
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    
    data = df[variables].dropna()
    result = coint_johansen(data, det_order=det_order, k_ar_diff=k_ar_diff)
    
    results_dict = {
        'trace_stat': result.trace_stat,
        'trace_crit_vals': result.trace_stat_crit_vals,
        'max_eig_stat': result.max_eig_stat,
        'max_eig_crit_vals': result.max_eig_stat_crit_vals,
        'eigenvalues': result.eig,
        'eigenvectors': result.evec,
    }
    
    return results_dict

def half_life(residuals):
    """
    Calculate half-life of mean reversion from residuals.
    
    Half-life = ln(2) / theta, where theta is the speed of mean reversion
    from the AR(1) process: delta_epsilon_t = theta * epsilon_{t-1} + noise
    
    Args:
        residuals: Residual series from cointegration regression
        
    Returns:
        Half-life in number of periods
    """
    # Estimate AR(1) coefficient
    res_series = pd.Series(residuals).dropna()
    res_lag = res_series.shift(1).dropna()
    res_diff = res_series.diff().dropna()
    
    # Align lengths
    common_idx = res_lag.index.intersection(res_diff.index)
    x = res_lag.loc[common_idx].values
    y = res_diff.loc[common_idx].values
    
    # OLS: delta_epsilon = theta * epsilon_{t-1} + error
    x_with_const = sm.add_constant(x)
    model = OLS(y, x_with_const)
    results = model.fit()
    
    theta = results.params[1]  # theta should be negative for mean reversion
    
    if theta >= 0:
        half_life_val = float('inf')  # No mean reversion
    else:
        half_life_val = -np.log(2) / theta
    
    return {
        'theta': theta,
        'half_life': half_life_val,
        'ar_coeff': 1 + theta,  # phi = 1 + theta
        'r_squared': results.rsquared,
    }

def hurst_exponent(series, max_lag=20):
    """
    Calculate Hurst exponent to determine if series is
    mean-reverting (H < 0.5), random walk (H = 0.5), or trending (H > 0.5).
    
    Args:
        series: Time series data
        max_lag: Maximum lag for R/S analysis
        
    Returns:
        Hurst exponent value
    """
    lags = range(2, max_lag)
    tau = [np.std(np.subtract(series[lag:], series[:-lag])) for lag in lags]
    
    # Linear regression on log-log scale
    reg = np.polyfit(np.log(lags), np.log(tau), 1)
    hurst = reg[0]
    
    return hurst

def variance_ratio_test(series, k=2):
    """
    Variance ratio test for random walk hypothesis.
    
    VR(k) = Var(r_k) / (k * Var(r_1))
    If VR = 1, series is a random walk
    If VR < 1, series is mean-reverting
    If VR > 1, series is trending
    
    Args:
        series: Price series
        k: Lag period
        
    Returns:
        Variance ratio and test statistic
    """
    returns = np.diff(series)
    n = len(returns)
    
    # 1-period returns
    var_1 = np.var(returns)
    
    # k-period returns
    returns_k = np.array([np.sum(returns[i:i+k]) for i in range(n - k + 1)])
    var_k = np.var(returns_k)
    
    vr = var_k / (k * var_1)
    
    # Lo-MacKinlay test statistic (homoskedastic)
    m = (n - k + 1) * (1 - k / n)
    vr_stat = (vr - 1) / np.sqrt(2 / m)
    
    return {
        'variance_ratio': vr,
        'test_statistic': vr_stat,
        'p_value': 2 * (1 - stats.norm.cdf(abs(vr_stat))),
        'is_mean_reverting': vr < 1,
        'is_trending': vr > 1,
    }

# Part 4: Module 3 - OU Mean Reversion Process

This module implements fitting of the Ornstein-Uhlenbeck (OU) process to describe mean reversion characteristics of the spread.

## 4.1 OU Process Theory
Discrete form of the OU process:
$$X_{t+1} - X_t = 	heta(\mu - X_t) + \sigma \cdot \epsilon$$

Where:
- $	heta$: Mean reversion speed parameter
- $\mu$: Long-term mean
- $\sigma$: Volatility parameter

## 4.2 Core Functionality
- **MLE Maximum Likelihood Estimation** - Estimate OU process parameters
- **Half-life Calculation** (CQF mandatory) - $	ext{half-life} = rac{\ln(2)}{	heta}$
- **Z-score Standardization** - For trading signal generation

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Ornstein-Uhlenbeck Process Fitting Module
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

def fit_ou_process(residuals, dt=1.0):
    """
    Fit an Ornstein-Uhlenbeck process to the residual series.
    
    OU SDE: dX_t = theta * (mu - X_t) * dt + sigma * dW_t
    
    Discrete form: X_{t+dt} - X_t = theta * (mu - X_t) * dt + sigma * sqrt(dt) * Z_t
    
    Args:
        residuals: Residual series from cointegration
        dt: Time step (default 1 day)
        
    Returns:
        Dictionary with OU parameters
    """
    x = np.array(residuals, dtype=float)
    n = len(x)
    
    # Method 1: Maximum Likelihood Estimation (MLE)
    # Log-likelihood function for OU process
    def neg_log_likelihood(params):
        theta, mu, sigma = params
        if theta <= 0 or sigma <= 0:
            return 1e10
        
        # Transition density is Gaussian
        # E[X_{t+dt} | X_t] = X_t * exp(-theta*dt) + mu * (1 - exp(-theta*dt))
        # Var[X_{t+dt} | X_t] = sigma^2 / (2*theta) * (1 - exp(-2*theta*dt))
        
        exp_theta_dt = np.exp(-theta * dt)
        mean = x[:-1] * exp_theta_dt + mu * (1 - exp_theta_dt)
        var = sigma**2 / (2 * theta) * (1 - np.exp(-2 * theta * dt))
        
        if var <= 0:
            return 1e10
        
        log_lik = -0.5 * np.sum(np.log(2 * np.pi * var) + (x[1:] - mean)**2 / var)
        return -log_lik
    
    # Initial guess from simple regression
    dx = np.diff(x)
    x_lag = x[:-1]
    beta_0 = np.mean(dx)
    beta_1 = np.cov(x_lag, dx)[0, 1] / np.var(x_lag)
    theta_init = -beta_1 / dt
    mu_init = beta_0 / (theta_init * dt) if theta_init != 0 else 0
    sigma_init = np.std(dx) / np.sqrt(dt)
    
    if theta_init <= 0:
        theta_init = 0.1
    if mu_init == 0:
        mu_init = np.mean(x)
    
    # MLE optimization
    x0 = [theta_init, mu_init, sigma_init]
    bounds = [(1e-6, None), (None, None), (1e-6, None)]
    
    result = minimize(neg_log_likelihood, x0, method='L-BFGS-B', bounds=bounds)
    
    theta_mle, mu_mle, sigma_mle = result.x
    
    # Method 2: Analytical solution (least squares)
    # dx_t = a + b * x_t + epsilon_t
    # where a = theta*mu*dt, b = -theta*dt
    x_with_const = np.column_stack([np.ones(n-1), x[:-1]])
    beta = np.linalg.lstsq(x_with_const, dx, rcond=None)[0]
    
    a_ls = beta[0]
    b_ls = beta[1]
    
    theta_ls = -b_ls / dt
    mu_ls = a_ls / (theta_ls * dt) if theta_ls != 0 else np.mean(x)
    residuals_ls = dx - (a_ls + b_ls * x[:-1])
    sigma_ls = np.std(residuals_ls) / np.sqrt(dt)
    
    # Calculate half-life
    if theta_mle > 0:
        half_life_mle = np.log(2) / theta_mle
    else:
        half_life_mle = float('inf')
    
    if theta_ls > 0:
        half_life_ls = np.log(2) / theta_ls
    else:
        half_life_ls = float('inf')
    
    # Long-term mean and equilibrium
    # Mean of stationary distribution = mu
    # Variance of stationary distribution = sigma^2 / (2*theta)
    if theta_mle > 0:
        var_stationary = sigma_mle**2 / (2 * theta_mle)
    else:
        var_stationary = np.var(x)
    
    std_stationary = np.sqrt(var_stationary)
    
    return {
        'theta_mle': theta_mle,
        'mu_mle': mu_mle,
        'sigma_mle': sigma_mle,
        'theta_ls': theta_ls,
        'mu_ls': mu_ls,
        'sigma_ls': sigma_ls,
        'half_life_mle': half_life_mle,
        'half_life_ls': half_life_ls,
        'mean_stationary': mu_mle,
        'std_stationary': std_stationary,
        'var_stationary': var_stationary,
        'log_likelihood': -result.fun,
        'converged': result.success,
    }

def ou_simulate(theta, mu, sigma, x0, n_steps, dt=1.0, seed=None):
    """
    Simulate an Ornstein-Uhlenbeck process.
    
    Args:
        theta: Speed of mean reversion
        mu: Long-term mean
        sigma: Volatility
        x0: Initial value
        n_steps: Number of time steps
        dt: Time step
        seed: Random seed
        
    Returns:
        Simulated OU process array
    """
    if seed is not None:
        np.random.seed(seed)
    
    x = np.zeros(n_steps)
    x[0] = x0
    
    exp_theta_dt = np.exp(-theta * dt)
    std_increment = sigma * np.sqrt((1 - np.exp(-2 * theta * dt)) / (2 * theta))
    
    for i in range(1, n_steps):
        x[i] = x[i-1] * exp_theta_dt + mu * (1 - exp_theta_dt) + std_increment * np.random.randn()
    
    return x

def ou_crossing_probability(x, threshold, theta, mu, sigma, dt=1.0):
    """
    Calculate probability of crossing threshold within one time step
    using the OU transition density.
    
    Args:
        x: Current value
        threshold: Threshold level
        theta: Speed of mean reversion
        mu: Long-term mean
        sigma: Volatility
        dt: Time step
        
    Returns:
        Probability of crossing threshold
    """
    exp_theta_dt = np.exp(-theta * dt)
    mean_next = x * exp_theta_dt + mu * (1 - exp_theta_dt)
    std_next = sigma * np.sqrt((1 - np.exp(-2 * theta * dt)) / (2 * theta))
    
    if x < threshold:
        prob = 1 - norm.cdf(threshold, loc=mean_next, scale=std_next)
    else:
        prob = norm.cdf(threshold, loc=mean_next, scale=std_next)
    
    return prob

def ou_expected_time_to_mean(x, theta, mu):
    """
    Expected time to reach mean level from current value.
    
    For OU process, expected time to reach mu from x is approximately:
    E[T] ≈ ln(|x - mu| / epsilon) / theta  (for small epsilon)
    
    More precisely, we use the first passage time approximation.
    
    Args:
        x: Current value
        theta: Speed of mean reversion
        mu: Long-term mean
        
    Returns:
        Expected time to reach mean (in dt units)
    """
    distance = abs(x - mu)
    if distance < 1e-10:
        return 0.0
    
    # Approximate expected time to revert to within 1% of distance
    epsilon = distance * 0.01
    expected_time = np.log(distance / epsilon) / theta
    
    return expected_time

def calculate_zscore(residuals, window=None):
    """
    Calculate z-score of residuals.
    
    Args:
        residuals: Residual series
        window: Rolling window size (None = use full sample)
        
    Returns:
        Z-score series
    """
    res_series = pd.Series(residuals)
    
    if window is None:
        mu = res_series.mean()
        sigma = res_series.std()
        zscore = (res_series - mu) / sigma
    else:
        rolling_mean = res_series.rolling(window=window).mean()
        rolling_std = res_series.rolling(window=window).std()
        zscore = (res_series - rolling_mean) / rolling_std
    
    return zscore.values

# Part 5: Module 4 - Trading Strategy and Threshold Optimization

This module implements the core logic of pairs trading strategy and threshold optimization.

## 5.1 Trading Rules
Symmetric trading rules based on Z-score:
- When $Z_t > Z_{open}$: Short the spread (short RB, long HC)
- When $Z_t < -Z_{open}$: Long the spread (long RB, short HC)
- When $|Z_t| < Z_{close}$: Close position

## 5.2 Threshold Optimization (CQF mandatory)
Iteratively optimize entry thresholds through grid search, with **Sharpe ratio maximization** as the objective.

Optimization range:
- Entry threshold: 1.0σ ~ 3.0σ (step size 0.2)
- Exit threshold: 0.5σ ~ 1.0σ (step size 0.1)

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Trading Strategy and Threshold Optimization Module
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd

class PairsTradingStrategy:
    """
    Pairs trading strategy based on cointegration residuals and z-score.
    
    Entry: z-score crosses entry threshold (open position)
    Exit: z-score reverts to mean / exit threshold (close position)
    Stop-loss: z-score exceeds stop-loss threshold (risk management)
    """
    
    def __init__(self, entry_z=2.0, exit_z=0.0, stop_loss_z=3.0, 
                 max_holding_days=30, transaction_cost=0.0005):
        """
        Initialize pairs trading strategy.
        
        Args:
            entry_z: Z-score entry threshold (open position when |z| > entry_z)
            exit_z: Z-score exit threshold (close position when |z| < exit_z)
            stop_loss_z: Z-score stop-loss threshold
            max_holding_days: Maximum holding period for a position
            transaction_cost: Transaction cost per trade (fraction of notional)
        """
        self.entry_z = entry_z
        self.exit_z = exit_z
        self.stop_loss_z = stop_loss_z
        self.max_holding_days = max_holding_days
        self.transaction_cost = transaction_cost
        
    def generate_signals(self, zscore, rb_price, hc_price, beta):
        """
        Generate trading signals from z-score series.
        
        Position types:
        - 0: No position
        - 1: Long RB, Short HC (spread is low, expect it to widen)
        - -1: Short RB, Long HC (spread is high, expect it to narrow)
        
        Args:
            zscore: Z-score of cointegration residuals
            rb_price: RB price series
            hc_price: HC price series
            beta: Hedge ratio (beta from cointegration regression)
            
        Returns:
            DataFrame with signals and positions
        """
        n = len(zscore)
        signals = pd.DataFrame({
            'zscore': zscore,
            'rb_price': rb_price.values if hasattr(rb_price, 'values') else rb_price,
            'hc_price': hc_price.values if hasattr(hc_price, 'values') else hc_price,
        })
        
        position = 0  # 0: flat, 1: long spread, -1: short spread
        entry_z_val = 0
        holding_days = 0
        positions = np.zeros(n)
        entry_dates = [None] * n
        exit_dates = [None] * n
        trade_types = [''] * n
        
        for i in range(n):
            z = zscore[i]
            
            if position == 0:
                # Look for entry signals
                if z > self.entry_z:
                    # Spread is too high: short RB, long HC
                    position = -1
                    entry_z_val = z
                    holding_days = 0
                    trade_types[i] = 'entry_short'
                elif z < -self.entry_z:
                    # Spread is too low: long RB, short HC
                    position = 1
                    entry_z_val = z
                    holding_days = 0
                    trade_types[i] = 'entry_long'
            else:
                holding_days += 1
                
                # Check exit conditions
                should_exit = False
                exit_reason = ''
                
                # Mean reversion exit
                if position == -1 and z < self.exit_z:
                    should_exit = True
                    exit_reason = 'mean_reversion'
                elif position == 1 and z > -self.exit_z:
                    should_exit = True
                    exit_reason = 'mean_reversion'
                
                # Stop loss
                if position == -1 and z > self.stop_loss_z:
                    should_exit = True
                    exit_reason = 'stop_loss'
                elif position == 1 and z < -self.stop_loss_z:
                    should_exit = True
                    exit_reason = 'stop_loss'
                
                # Max holding period
                if holding_days >= self.max_holding_days:
                    should_exit = True
                    exit_reason = 'max_holding'
                
                if should_exit:
                    trade_types[i] = f'exit_{exit_reason}'
                    position = 0
                    holding_days = 0
            
            positions[i] = position
        
        signals['position'] = positions
        signals['trade_type'] = trade_types
        signals['holding_days'] = 0
        
        # Calculate holding days for each position
        current_hold = 0
        for i in range(n):
            if positions[i] != 0:
                current_hold += 1
                signals.loc[i, 'holding_days'] = current_hold
            else:
                current_hold = 0
        
        return signals

def optimize_threshold(zscore, rb_price, hc_price, beta, 
                       entry_range=(1.0, 3.0), exit_range=(0.0, 1.0),
                       step=0.1, metric='sharpe'):
    """
    Grid search for optimal entry and exit z-score thresholds.
    
    Args:
        zscore: Z-score series
        rb_price: RB price series
        hc_price: HC price series
        beta: Hedge ratio
        entry_range: Range of entry thresholds to test
        exit_range: Range of exit thresholds to test
        step: Step size for grid search
        metric: Optimization metric ('sharpe', 'total_return', 'win_rate')
        
    Returns:
        Dictionary with optimization results
    """
    from backtest import BacktestEngine
    
    entry_values = np.arange(entry_range[0], entry_range[1] + step, step)
    exit_values = np.arange(exit_range[0], exit_range[1] + step, step)
    
    results = []
    
    for entry_z in entry_values:
        for exit_z in exit_values:
            if exit_z >= entry_z:
                continue
                
            strategy = PairsTradingStrategy(
                entry_z=entry_z, 
                exit_z=exit_z,
                stop_loss_z=entry_z + 1.0,
                max_holding_days=60
            )
            
            signals = strategy.generate_signals(zscore, rb_price, hc_price, beta)
            
            engine = BacktestEngine(initial_capital=1000000, transaction_cost=0.0005)
            bt_result = engine.run_backtest(signals, beta)
            
            results.append({
                'entry_z': entry_z,
                'exit_z': exit_z,
                'total_return': bt_result['total_return'],
                'sharpe_ratio': bt_result['sharpe_ratio'],
                'max_drawdown': bt_result['max_drawdown'],
                'num_trades': bt_result['num_trades'],
                'win_rate': bt_result['win_rate'],
                'profit_factor': bt_result['profit_factor'],
            })
    
    results_df = pd.DataFrame(results)
    
    # Find best parameters
    if metric == 'sharpe':
        best_idx = results_df['sharpe_ratio'].idxmax()
    elif metric == 'total_return':
        best_idx = results_df['total_return'].idxmax()
    elif metric == 'win_rate':
        best_idx = results_df['win_rate'].idxmax()
    else:
        best_idx = results_df['sharpe_ratio'].idxmax()
    
    best_params = results_df.iloc[best_idx].to_dict()
    
    return {
        'all_results': results_df,
        'best_params': best_params,
        'best_entry_z': best_params['entry_z'],
        'best_exit_z': best_params['exit_z'],
        'metric': metric,
    }

def calculate_position_sizing(capital, rb_price, hc_price, beta, position_type):
    """
    Calculate position sizes for both legs of the pair trade.
    
    For a dollar-neutral pair:
    - Value of RB position = capital / 2
    - Value of HC position = capital / 2
    
    Adjusted by hedge ratio beta:
    - Number of RB contracts = (capital / 2) / rb_price
    - Number of HC contracts = beta * (capital / 2) / hc_price
    
    Args:
        capital: Total capital allocated
        rb_price: Current RB price
        hc_price: Current HC price
        beta: Hedge ratio
        position_type: 1 (long RB, short HC) or -1 (short RB, long HC)
        
    Returns:
        Dictionary with position sizes
    """
    capital_per_leg = capital / 2
    
    rb_contracts = capital_per_leg / rb_price
    hc_contracts = beta * capital_per_leg / hc_price
    
    if position_type == 1:
        # Long RB, Short HC
        rb_position = rb_contracts
        hc_position = -hc_contracts
    else:
        # Short RB, Long HC
        rb_position = -rb_contracts
        hc_position = hc_contracts
    
    return {
        'rb_contracts': rb_position,
        'hc_contracts': hc_position,
        'rb_notional': abs(rb_position) * rb_price,
        'hc_notional': abs(hc_position) * hc_price,
        'total_notional': abs(rb_position) * rb_price + abs(hc_position) * hc_price,
    }

# Part 6: Module 5 - Backtesting Engine

This module implements a complete backtesting system to calculate strategy performance metrics.

## 6.1 Performance Metrics
- **Return Metrics**: Total return, annualized return, cumulative return
- **Risk Metrics**: Volatility, maximum drawdown, downside volatility
- **Risk-adjusted Returns**: Sharpe ratio, Sortino ratio, Calmar ratio
- **Trading Statistics**: Number of trades, win rate, average holding period

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Backtesting Engine Module (Revised)
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd

class BacktestEngine:
    """
    Backtesting engine for pairs trading strategy.
    
    Features:
    - Daily mark-to-market P&L calculation
    - Transaction cost modeling
    - Position sizing based on hedge ratio
    - Comprehensive performance metrics
    - Trade-by-trade P&L tracking
    """
    
    def __init__(self, initial_capital=1000000, transaction_cost=0.0005, 
                 contract_multiplier_rb=10, contract_multiplier_hc=10):
        """
        Initialize backtest engine.
        
        Args:
            initial_capital: Initial capital in RMB
            transaction_cost: Transaction cost per trade (fraction of notional)
            contract_multiplier_rb: Contract multiplier for RB (10 tons/contract)
            contract_multiplier_hc: Contract multiplier for HC (10 tons/contract)
        """
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost
        self.contract_multiplier_rb = contract_multiplier_rb
        self.contract_multiplier_hc = contract_multiplier_hc
        
    def run_backtest(self, signals, beta):
        """
        Run backtest on trading signals.
        
        Args:
            signals: DataFrame with position, rb_price, hc_price columns
            beta: Hedge ratio from cointegration regression
            
        Returns:
            Dictionary with backtest results
        """
        n = len(signals)
        
        # Initialize tracking arrays
        portfolio_value = np.zeros(n)
        daily_pnl = np.zeros(n)
        rb_contracts = np.zeros(n)  # Number of contracts (positive=long, negative=short)
        hc_contracts = np.zeros(n)
        cash = np.zeros(n)
        
        # Trade tracking
        trades = []
        current_trade = None
        trade_count = 0
        
        # State variables
        current_cash = self.initial_capital
        current_rb = 0.0  # RB contracts held
        current_hc = 0.0  # HC contracts held
        prev_rb_price = 0
        prev_hc_price = 0
        
        for i in range(n):
            rb_price = float(signals['rb_price'].iloc[i])
            hc_price = float(signals['hc_price'].iloc[i])
            pos = int(signals['position'].iloc[i])
            trade_type = str(signals['trade_type'].iloc[i])
            
            # Calculate daily P&L from position marking
            if i > 0 and current_rb != 0:
                rb_pnl = current_rb * (rb_price - prev_rb_price) * self.contract_multiplier_rb
                hc_pnl = current_hc * (hc_price - prev_hc_price) * self.contract_multiplier_hc
                daily_pnl[i] = rb_pnl + hc_pnl
                current_cash += daily_pnl[i]
            
            # Check for entry signal
            if trade_type.startswith('entry') and current_rb == 0:
                trade_count += 1
                
                # Calculate position sizes (dollar neutral, 50% of cash per leg notional)
                notional_per_leg = current_cash * 0.4  # Use 40% per leg = 80% total, leave 20% buffer
                
                if trade_type == 'entry_long':
                    # Long RB, Short HC
                    rb_qty = notional_per_leg / (rb_price * self.contract_multiplier_rb)
                    hc_qty = beta * notional_per_leg / (hc_price * self.contract_multiplier_hc)
                    
                    current_rb = rb_qty
                    current_hc = -hc_qty
                else:
                    # Short RB, Long HC
                    rb_qty = notional_per_leg / (rb_price * self.contract_multiplier_rb)
                    hc_qty = beta * notional_per_leg / (hc_price * self.contract_multiplier_hc)
                    
                    current_rb = -rb_qty
                    current_hc = hc_qty
                
                # Transaction cost for opening
                notional_rb = abs(current_rb) * rb_price * self.contract_multiplier_rb
                notional_hc = abs(current_hc) * hc_price * self.contract_multiplier_hc
                open_cost = (notional_rb + notional_hc) * self.transaction_cost
                current_cash -= open_cost
                
                # Record trade entry
                current_trade = {
                    'entry_idx': i,
                    'entry_rb_price': rb_price,
                    'entry_hc_price': hc_price,
                    'rb_contracts': current_rb,
                    'hc_contracts': current_hc,
                    'direction': 'long' if 'long' in trade_type else 'short',
                    'open_cost': open_cost,
                    'entry_zscore': float(signals['zscore'].iloc[i]) if 'zscore' in signals.columns else 0,
                }
            
            # Check for exit signal
            elif trade_type.startswith('exit') and current_rb != 0:
                # Transaction cost for closing
                notional_rb = abs(current_rb) * rb_price * self.contract_multiplier_rb
                notional_hc = abs(current_hc) * hc_price * self.contract_multiplier_hc
                close_cost = (notional_rb + notional_hc) * self.transaction_cost
                current_cash -= close_cost
                
                # Calculate trade P&L
                if current_trade is not None:
                    rb_pnl_total = current_rb * (rb_price - current_trade['entry_rb_price']) * self.contract_multiplier_rb
                    hc_pnl_total = current_hc * (hc_price - current_trade['entry_hc_price']) * self.contract_multiplier_hc
                    total_pnl = rb_pnl_total + hc_pnl_total - current_trade['open_cost'] - close_cost
                    
                    current_trade['exit_idx'] = i
                    current_trade['exit_rb_price'] = rb_price
                    current_trade['exit_hc_price'] = hc_price
                    current_trade['exit_reason'] = trade_type.replace('exit_', '')
                    current_trade['total_pnl'] = total_pnl
                    current_trade['close_cost'] = close_cost
                    current_trade['holding_days'] = i - current_trade['entry_idx']
                    current_trade['is_win'] = total_pnl > 0
                    current_trade['exit_zscore'] = float(signals['zscore'].iloc[i]) if 'zscore' in signals.columns else 0
                    
                    trades.append(current_trade)
                    current_trade = None
                
                # Close positions
                current_rb = 0
                current_hc = 0
            
            # Update portfolio value
            rb_mtm = current_rb * rb_price * self.contract_multiplier_rb
            hc_mtm = current_hc * hc_price * self.contract_multiplier_hc
            portfolio_value[i] = current_cash + rb_mtm + hc_mtm
            
            # Store values
            rb_contracts[i] = current_rb
            hc_contracts[i] = current_hc
            cash[i] = current_cash
            
            prev_rb_price = rb_price
            prev_hc_price = hc_price
        
        # Calculate performance metrics
        portfolio_series = pd.Series(portfolio_value)
        daily_returns = portfolio_series.pct_change().dropna()
        
        # Total return
        total_return = (portfolio_value[-1] - self.initial_capital) / self.initial_capital
        
        # Annualized return (assuming 252 trading days)
        n_days = len(portfolio_value)
        ann_return = (1 + total_return) ** (252 / n_days) - 1 if n_days > 0 else 0
        
        # Annualized volatility
        ann_vol = daily_returns.std() * np.sqrt(252) if len(daily_returns) > 0 else 0
        
        # Sharpe ratio (assuming 0 risk-free rate)
        sharpe_ratio = ann_return / ann_vol if ann_vol > 0 else 0
        
        # Max drawdown
        cummax = portfolio_series.cummax()
        drawdown = (portfolio_series - cummax) / cummax
        max_drawdown = drawdown.min()
        
        # Win rate
        wins = sum(1 for t in trades if t['is_win'])
        losses = sum(1 for t in trades if not t['is_win'])
        win_rate = wins / len(trades) if len(trades) > 0 else 0
        
        # Profit factor
        total_profit = sum(t['total_pnl'] for t in trades if t['is_win'])
        total_loss = sum(abs(t['total_pnl']) for t in trades if not t['is_win'])
        profit_factor = total_profit / total_loss if total_loss > 0 else float('inf')
        
        # Calmar ratio
        calmar_ratio = ann_return / abs(max_drawdown) if max_drawdown != 0 else 0
        
        # Sortino ratio (downside deviation)
        downside_returns = daily_returns[daily_returns < 0]
        downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
        sortino_ratio = ann_return / downside_vol if downside_vol > 0 else 0
        
        results = {
            'portfolio_value': portfolio_value,
            'daily_pnl': daily_pnl,
            'daily_returns': daily_returns.values,
            'rb_position': rb_contracts,
            'hc_position': hc_contracts,
            'cash': cash,
            'drawdown': drawdown.values,
            'trades': trades,
            'total_return': total_return,
            'annualized_return': ann_return,
            'annualized_volatility': ann_vol,
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'calmar_ratio': calmar_ratio,
            'max_drawdown': max_drawdown,
            'num_trades': len(trades),
            'wins': wins,
            'losses': losses,
            'win_rate': win_rate,
            'profit_factor': profit_factor,
            'total_profit': total_profit,
            'total_loss': total_loss,
            'initial_capital': self.initial_capital,
            'final_capital': portfolio_value[-1],
            'n_days': n_days,
        }
        
        return results

def calculate_trade_analytics(signals, beta, contract_multiplier_rb=10, contract_multiplier_hc=10):
    """
    Calculate detailed trade-by-trade analytics.
    
    Args:
        signals: DataFrame with trading signals
        beta: Hedge ratio
        contract_multiplier_rb: RB contract multiplier
        contract_multiplier_hc: HC contract multiplier
        
    Returns:
        DataFrame with individual trade details
    """
    engine = BacktestEngine(
        contract_multiplier_rb=contract_multiplier_rb,
        contract_multiplier_hc=contract_multiplier_hc
    )
    bt_result = engine.run_backtest(signals, beta)
    
    trades = bt_result['trades']
    if not trades:
        return pd.DataFrame()
    
    trades_df = pd.DataFrame(trades)
    
    # Add return percentage
    if 'entry_rb_price' in trades_df.columns:
        entry_notional = (trades_df['entry_rb_price'] * contract_multiplier_rb * abs(trades_df['rb_contracts']) +
                         trades_df['entry_hc_price'] * contract_multiplier_hc * abs(trades_df['hc_contracts']))
        trades_df['return_pct'] = trades_df['total_pnl'] / entry_notional * 100
    
    return trades_df

# Part 7: Module 6 - Rolling Window Dynamic Analysis

This module implements rolling window cointegration analysis to verify the stability of cointegration relationships.

## 7.1 Rolling Window Settings
- **Window length**: 8 months (approx 160 trading days)
- **Rolling step size**: 10 trading days (high-frequency rolling)
- **Total windows**: Approx 559 valid windows

## 7.2 Analysis Content
- Dynamic hedge ratio evolution
- Cointegration test pass rate
- Structural break identification
- Dynamic strategy performance comparison

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Rolling Window Dynamic Cointegration Module
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd
from cointegration import engle_granger_cointegration, adf_test
from ou_process import calculate_zscore

class RollingCointegration:
    """
    Rolling window cointegration analysis for dynamic pairs trading.
    
    Features:
    - Rolling estimation of cointegration beta
    - Rolling ADF test for cointegration stability
    - Dynamic z-score calculation
    - Cointegration break detection
    """
    
    def __init__(self, window_size=168, step=1):
        """
        Initialize rolling cointegration analyzer.
        
        Args:
            window_size: Rolling window size in days (default 168 ≈ 8 months)
            step: Step size for rolling window (default 1 day)
        """
        self.window_size = window_size
        self.step = step
        
    def fit(self, rb_log, hc_log, dates=None):
        """
        Run rolling cointegration analysis.
        
        Args:
            rb_log: RB log price series (array or Series)
            hc_log: HC log price series (array or Series)
            dates: Date index (optional)
            
        Returns:
            DataFrame with rolling cointegration results
        """
        n = len(rb_log)
        rb_arr = np.asarray(rb_log, dtype=float)
        hc_arr = np.asarray(hc_log, dtype=float)
        
        results = []
        
        for i in range(self.window_size, n, self.step):
            start_idx = i - self.window_size
            end_idx = i
            
            window_rb = rb_arr[start_idx:end_idx]
            window_hc = hc_arr[start_idx:end_idx]
            
            try:
                eg_result = engle_granger_cointegration(window_rb, window_hc)
                
                result = {
                    'end_idx': end_idx,
                    'end_date': dates.iloc[end_idx] if dates is not None else end_idx,
                    'alpha': eg_result['alpha'],
                    'beta': eg_result['beta'],
                    'residual_std': eg_result['residual_std'],
                    'adf_statistic': eg_result['adf_statistic'],
                    'adf_pvalue': eg_result['adf_pvalue'],
                    'is_cointegrated_5pct': eg_result['is_cointegrated_5pct'],
                    'r_squared': eg_result['r_squared'],
                }
                results.append(result)
            except Exception as e:
                print(f"Warning: Rolling window at index {i} failed: {e}")
                continue
        
        results_df = pd.DataFrame(results)
        
        return results_df
    
    def generate_dynamic_signals(self, rb_log, hc_log, rb_price, hc_price,
                                  entry_z=2.0, exit_z=0.0, stop_loss_z=3.0,
                                  max_holding_days=30, transaction_cost=0.0005):
        """
        Generate trading signals using dynamically estimated beta and z-score.
        
        Args:
            rb_log: RB log price series
            hc_log: HC log price series
            rb_price: RB price series
            hc_price: HC price series
            entry_z: Entry z-score threshold
            exit_z: Exit z-score threshold
            stop_loss_z: Stop loss z-score threshold
            max_holding_days: Maximum holding days
            transaction_cost: Transaction cost
            
        Returns:
            Dictionary with signals and rolling results
        """
        n = len(rb_log)
        rb_arr = np.asarray(rb_log, dtype=float)
        hc_arr = np.asarray(hc_log, dtype=float)
        rb_price_arr = np.asarray(rb_price, dtype=float)
        hc_price_arr = np.asarray(hc_price, dtype=float)
        
        # Initialize arrays
        rolling_beta = np.full(n, np.nan)
        rolling_alpha = np.full(n, np.nan)
        rolling_residual_std = np.full(n, np.nan)
        rolling_zscore = np.full(n, np.nan)
        rolling_adf_pvalue = np.full(n, np.nan)
        is_cointegrated = np.full(n, False)
        
        position = 0
        positions = np.zeros(n)
        holding_days_arr = np.zeros(n)
        trade_types = [''] * n
        current_hold = 0
        
        for i in range(self.window_size, n):
            start_idx = i - self.window_size
            window_rb = rb_arr[start_idx:i]
            window_hc = hc_arr[start_idx:i]
            
            try:
                eg_result = engle_granger_cointegration(window_rb, window_hc)
                
                rolling_beta[i] = eg_result['beta']
                rolling_alpha[i] = eg_result['alpha']
                rolling_residual_std[i] = eg_result['residual_std']
                rolling_adf_pvalue[i] = eg_result['adf_pvalue']
                is_cointegrated[i] = eg_result['is_cointegrated_5pct']
                
                # Calculate current z-score using rolling parameters
                current_residual = rb_arr[i] - eg_result['alpha'] - eg_result['beta'] * hc_arr[i]
                rolling_zscore[i] = current_residual / eg_result['residual_std']
                
            except Exception as e:
                rolling_zscore[i] = np.nan
                continue
            
            z = rolling_zscore[i]
            if np.isnan(z):
                positions[i] = position
                if position != 0:
                    current_hold += 1
                    holding_days_arr[i] = current_hold
                continue
            
            if position == 0:
                # Only enter if cointegrated
                if is_cointegrated[i]:
                    if z > entry_z:
                        position = -1
                        current_hold = 0
                        trade_types[i] = 'entry_short'
                    elif z < -entry_z:
                        position = 1
                        current_hold = 0
                        trade_types[i] = 'entry_long'
            else:
                current_hold += 1
                holding_days_arr[i] = current_hold
                
                should_exit = False
                exit_reason = ''
                
                # Mean reversion exit
                if position == -1 and z < exit_z:
                    should_exit = True
                    exit_reason = 'mean_reversion'
                elif position == 1 and z > -exit_z:
                    should_exit = True
                    exit_reason = 'mean_reversion'
                
                # Stop loss
                if position == -1 and z > stop_loss_z:
                    should_exit = True
                    exit_reason = 'stop_loss'
                elif position == 1 and z < -stop_loss_z:
                    should_exit = True
                    exit_reason = 'stop_loss'
                
                # Max holding period
                if current_hold >= max_holding_days:
                    should_exit = True
                    exit_reason = 'max_holding'
                
                # If cointegration breaks, exit
                if not is_cointegrated[i]:
                    should_exit = True
                    exit_reason = 'cointegration_break'
                
                if should_exit:
                    trade_types[i] = f'exit_{exit_reason}'
                    position = 0
                    current_hold = 0
            
            positions[i] = position
        
        signals = pd.DataFrame({
            'zscore': rolling_zscore,
            'rb_price': rb_price_arr,
            'hc_price': hc_price_arr,
            'position': positions,
            'holding_days': holding_days_arr,
            'trade_type': trade_types,
            'rolling_beta': rolling_beta,
            'rolling_alpha': rolling_alpha,
            'rolling_residual_std': rolling_residual_std,
            'rolling_adf_pvalue': rolling_adf_pvalue,
            'is_cointegrated': is_cointegrated,
        })
        
        return {
            'signals': signals,
            'rolling_beta': rolling_beta,
            'rolling_alpha': rolling_alpha,
            'rolling_residual_std': rolling_residual_std,
            'rolling_adf_pvalue': rolling_adf_pvalue,
            'rolling_zscore': rolling_zscore,
            'is_cointegrated': is_cointegrated,
        }

def structural_break_analysis(residuals, dates=None, window_size=60):
    """
    Analyze structural breaks in the cointegration relationship.
    
    Args:
        residuals: Full sample residuals
        dates: Date index
        window_size: Rolling window for break detection
        
    Returns:
        Dictionary with structural break analysis results
    """
    res_series = pd.Series(residuals)
    n = len(res_series)
    
    # Rolling ADF test
    rolling_adf = []
    rolling_pvalues = []
    
    for i in range(window_size, n):
        window = res_series.iloc[i-window_size:i]
        try:
            adf_result = adf_test(window)
            rolling_adf.append(adf_result['adf_statistic'])
            rolling_pvalues.append(adf_result['p_value'])
        except:
            rolling_adf.append(np.nan)
            rolling_pvalues.append(np.nan)
    
    # Rolling mean and std
    rolling_mean = res_series.rolling(window=window_size).mean()
    rolling_std = res_series.rolling(window=window_size).std()
    
    # Detect breaks: periods where cointegration fails (p > 0.05)
    pvalue_series = pd.Series(rolling_pvalues, index=res_series.index[window_size:])
    break_periods = pvalue_series[pvalue_series > 0.05]
    
    results = {
        'rolling_adf': rolling_adf,
        'rolling_pvalues': rolling_pvalues,
        'rolling_mean': rolling_mean.values,
        'rolling_std': rolling_std.values,
        'break_periods': break_periods,
        'num_break_days': len(break_periods),
        'break_ratio': len(break_periods) / (n - window_size) if n > window_size else 0,
    }
    
    return results

# Part 8: Module 7 - Visualization Analysis

This module generates all analysis charts, including:
- Price trends and spread charts
- Cointegration residuals and Z-score charts
- Trading signals and position charts
- Cumulative returns and drawdown charts
- Rolling window analysis charts

In [ ]:
"""
CQF Final Project - TS: Pairs Trading
Visualization Module
Author: CQF Candidate
Date: 2026
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import os

# Set Chinese font support
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

def plot_price_series(df, save_path=None):
    """Plot RB and HC price series."""
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(df['date'], df['rb_close'], label='RB (Rebar)', color='#1f77b4', linewidth=1)
    ax.plot(df['date'], df['hc_close'], label='HC (Hot-Rolled Coil)', color='#ff7f0e', linewidth=1)
    
    ax.set_title('RB vs HC Futures Price (2023-2025)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (RMB/ton)')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_log_prices(df, save_path=None):
    """Plot log prices."""
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(df['date'], df['rb_log'], label='RB Log Price', color='#1f77b4', linewidth=1)
    ax.plot(df['date'], df['hc_log'], label='HC Log Price', color='#ff7f0e', linewidth=1)
    
    ax.set_title('Log Price Series', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Log Price')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_spread(df, eg_result, save_path=None):
    """Plot the cointegration spread/residuals."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Price spread
    ax1.plot(df['date'], df['spread_price'], color='#2ca02c', linewidth=1)
    ax1.axhline(y=np.mean(df['spread_price']), color='red', linestyle='--', alpha=0.7, label='Mean')
    ax1.set_title('Price Spread (RB - HC)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Spread (RMB/ton)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Cointegration residuals
    residuals = eg_result['residuals']
    zscore = (residuals - np.mean(residuals)) / np.std(residuals)
    
    ax2.plot(df['date'], residuals, color='#9467bd', linewidth=1)
    ax2.axhline(y=eg_result['alpha'], color='red', linestyle='--', alpha=0.7, label='Equilibrium')
    ax2.axhline(y=eg_result['alpha'] + eg_result['residual_std'], color='orange', linestyle=':', alpha=0.7, label='+1σ')
    ax2.axhline(y=eg_result['alpha'] - eg_result['residual_std'], color='orange', linestyle=':', alpha=0.7, label='-1σ')
    ax2.axhline(y=eg_result['alpha'] + 2*eg_result['residual_std'], color='red', linestyle=':', alpha=0.5, label='+2σ')
    ax2.axhline(y=eg_result['alpha'] - 2*eg_result['residual_std'], color='red', linestyle=':', alpha=0.5, label='-2σ')
    ax2.set_title('Cointegration Residuals (Engle-Granger)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Residual')
    ax2.legend(loc='best', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_zscore(zscore, dates, entry_z=2.0, exit_z=0.0, save_path=None):
    """Plot z-score with entry/exit thresholds."""
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(dates, zscore, color='#17becf', linewidth=1, label='Z-score')
    ax.axhline(y=entry_z, color='red', linestyle='--', alpha=0.7, label=f'Entry (+{entry_z}σ)')
    ax.axhline(y=-entry_z, color='red', linestyle='--', alpha=0.7, label=f'Entry (-{entry_z}σ)')
    ax.axhline(y=exit_z, color='green', linestyle='--', alpha=0.7, label=f'Exit ({exit_z}σ)')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Shade entry zones
    ax.fill_between(dates, entry_z, max(zscore), where=zscore > entry_z, 
                    color='red', alpha=0.1, label='Short Zone')
    ax.fill_between(dates, min(zscore), -entry_z, where=zscore < -entry_z, 
                    color='green', alpha=0.1, label='Long Zone')
    
    ax.set_title('Z-Score of Cointegration Residuals', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Z-Score')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_backtest_results(bt_result, dates, save_path=None):
    """Plot backtest results: portfolio value, drawdown, daily P&L."""
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(3, 1, height_ratios=[3, 1, 1], hspace=0.3)
    
    # Portfolio value
    ax1 = fig.add_subplot(gs[0])
    portfolio_val = bt_result['portfolio_value']
    ax1.plot(dates, portfolio_val, color='#1f77b4', linewidth=1.2, label='Portfolio Value')
    ax1.axhline(y=bt_result['initial_capital'], color='red', linestyle='--', alpha=0.5, label='Initial Capital')
    ax1.set_title('Portfolio Value', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Value (RMB)')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    
    # Drawdown
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    drawdown = bt_result['drawdown'] * 100
    ax2.fill_between(dates, drawdown, 0, color='red', alpha=0.3)
    ax2.plot(dates, drawdown, color='red', linewidth=0.8)
    ax2.set_title('Drawdown', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Drawdown (%)')
    ax2.grid(True, alpha=0.3)
    
    # Daily P&L
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    daily_pnl = bt_result['daily_pnl']
    colors = ['green' if p >= 0 else 'red' for p in daily_pnl]
    ax3.bar(dates, daily_pnl, color=colors, alpha=0.6, width=1)
    ax3.set_title('Daily P&L', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('P&L (RMB)')
    ax3.grid(True, alpha=0.3)
    
    plt.setp(ax1.get_xticklabels(), rotation=45)
    plt.setp(ax2.get_xticklabels(), rotation=45)
    plt.setp(ax3.get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_trade_signals(signals, dates, save_path=None):
    """Plot z-score with entry/exit signals marked."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(dates, signals['zscore'], color='#17becf', linewidth=1, label='Z-score', alpha=0.8)
    ax.axhline(y=2, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=-2, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Mark entry and exit points
    entry_long = signals[signals['trade_type'] == 'entry_long']
    entry_short = signals[signals['trade_type'] == 'entry_short']
    exit_long = signals[signals['trade_type'].str.startswith('exit') & (signals['position'].shift(1) == 1)]
    exit_short = signals[signals['trade_type'].str.startswith('exit') & (signals['position'].shift(1) == -1)]
    
    if len(entry_long) > 0:
        ax.scatter(dates[entry_long.index], entry_long['zscore'], 
                   color='green', marker='^', s=80, label='Entry Long', zorder=5)
    if len(entry_short) > 0:
        ax.scatter(dates[entry_short.index], entry_short['zscore'], 
                   color='red', marker='v', s=80, label='Entry Short', zorder=5)
    
    # Shade position periods
    position = signals['position'].values
    ax.fill_between(dates, ax.get_ylim()[0], ax.get_ylim()[1], 
                    where=position != 0, color='yellow', alpha=0.1, label='In Position')
    
    ax.set_title('Trading Signals on Z-Score', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Z-Score')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_rolling_beta(rolling_result, dates, save_path=None):
    """Plot rolling beta over time."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    valid_idx = ~np.isnan(rolling_result['rolling_beta'])
    valid_dates = dates[valid_idx]
    valid_beta = rolling_result['rolling_beta'][valid_idx]
    
    ax1.plot(valid_dates, valid_beta, color='#9467bd', linewidth=1.2)
    ax1.axhline(y=np.mean(valid_beta), color='red', linestyle='--', alpha=0.7, label=f'Mean = {np.mean(valid_beta):.4f}')
    ax1.set_title('Rolling Cointegration Beta (8-month window)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Beta (Hedge Ratio)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Cointegration stability
    valid_pvalue = rolling_result['rolling_adf_pvalue'][valid_idx]
    ax2.plot(valid_dates, valid_pvalue, color='#d62728', linewidth=1)
    ax2.axhline(y=0.05, color='green', linestyle='--', alpha=0.7, label='5% Significance')
    ax2.fill_between(valid_dates, 0, 0.05, where=valid_pvalue < 0.05, 
                     color='green', alpha=0.1, label='Cointegrated')
    ax2.fill_between(valid_dates, 0.05, 1, where=valid_pvalue >= 0.05, 
                     color='red', alpha=0.1, label='Not Cointegrated')
    ax2.set_title('Rolling ADF Test P-Value', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('P-Value')
    ax2.set_ylim(0, 0.5)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_threshold_heatmap(opt_result, save_path=None):
    """Plot threshold optimization heatmap."""
    results_df = opt_result['all_results']
    
    # Pivot data for heatmap
    pivot_sharpe = results_df.pivot(index='entry_z', columns='exit_z', values='sharpe_ratio')
    pivot_return = results_df.pivot(index='entry_z', columns='exit_z', values='total_return')
    pivot_trades = results_df.pivot(index='entry_z', columns='exit_z', values='num_trades')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Sharpe ratio heatmap
    im1 = axes[0].imshow(pivot_sharpe.values, cmap='RdYlGn', aspect='auto')
    axes[0].set_title('Sharpe Ratio', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Exit Z-Score')
    axes[0].set_ylabel('Entry Z-Score')
    axes[0].set_xticks(range(len(pivot_sharpe.columns)))
    axes[0].set_xticklabels([f'{x:.1f}' for x in pivot_sharpe.columns])
    axes[0].set_yticks(range(len(pivot_sharpe.index)))
    axes[0].set_yticklabels([f'{x:.1f}' for x in pivot_sharpe.index])
    plt.colorbar(im1, ax=axes[0])
    
    # Mark best
    best_entry = opt_result['best_entry_z']
    best_exit = opt_result['best_exit_z']
    entry_idx = list(pivot_sharpe.index).index(best_entry)
    exit_idx = list(pivot_sharpe.columns).index(best_exit)
    axes[0].plot(exit_idx, entry_idx, 'k*', markersize=15, label='Best')
    
    # Total return heatmap
    im2 = axes[1].imshow(pivot_return.values * 100, cmap='RdYlGn', aspect='auto')
    axes[1].set_title('Total Return (%)', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Exit Z-Score')
    axes[1].set_ylabel('Entry Z-Score')
    axes[1].set_xticks(range(len(pivot_return.columns)))
    axes[1].set_xticklabels([f'{x:.1f}' for x in pivot_return.columns])
    axes[1].set_yticks(range(len(pivot_return.index)))
    axes[1].set_yticklabels([f'{x:.1f}' for x in pivot_return.index])
    plt.colorbar(im2, ax=axes[1])
    
    # Number of trades heatmap
    im3 = axes[2].imshow(pivot_trades.values, cmap='YlOrRd', aspect='auto')
    axes[2].set_title('Number of Trades', fontsize=11, fontweight='bold')
    axes[2].set_xlabel('Exit Z-Score')
    axes[2].set_ylabel('Entry Z-Score')
    axes[2].set_xticks(range(len(pivot_trades.columns)))
    axes[2].set_xticklabels([f'{x:.1f}' for x in pivot_trades.columns])
    axes[2].set_yticks(range(len(pivot_trades.index)))
    axes[2].set_yticklabels([f'{x:.1f}' for x in pivot_trades.index])
    plt.colorbar(im3, ax=axes[2])
    
    plt.suptitle('Threshold Optimization Grid Search', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_ou_fit(residuals, ou_params, save_path=None):
    """Plot OU process fit diagnostics."""
    from ou_process import ou_simulate
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Residuals vs OU simulation
    ax1 = axes[0, 0]
    ax1.plot(residuals, color='#1f77b4', linewidth=0.8, alpha=0.7, label='Actual Residuals')
    
    # Simulate OU with fitted parameters
    n_sim = len(residuals)
    sim = ou_simulate(ou_params['theta_mle'], ou_params['mu_mle'], ou_params['sigma_mle'],
                      residuals[0], n_sim, seed=42)
    ax1.plot(sim, color='red', linewidth=0.8, alpha=0.7, label='OU Simulation')
    ax1.set_title('Actual vs OU Simulated Residuals', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Time (days)')
    ax1.set_ylabel('Residual')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Distribution of residuals
    ax2 = axes[0, 1]
    ax2.hist(residuals, bins=50, density=True, alpha=0.6, color='#2ca02c', label='Empirical')
    
    # Theoretical stationary distribution
    from scipy.stats import norm
    x_range = np.linspace(min(residuals), max(residuals), 200)
    pdf = norm.pdf(x_range, loc=ou_params['mu_mle'], scale=ou_params['std_stationary'])
    ax2.plot(x_range, pdf, 'r-', linewidth=2, label='OU Stationary Dist.')
    ax2.set_title('Residual Distribution', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Residual')
    ax2.set_ylabel('Density')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Autocorrelation
    ax3 = axes[1, 0]
    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(residuals, lags=50, ax=ax3, alpha=0.05)
    ax3.set_title('Autocorrelation Function', fontsize=11, fontweight='bold')
    ax3.set_xlabel('Lag (days)')
    ax3.grid(True, alpha=0.3)
    
    # Mean reversion speed visualization
    ax4 = axes[1, 1]
    half_life = ou_params['half_life_mle']
    distances = np.linspace(0.1, 3, 100)
    times = np.log(distances / 0.01) / ou_params['theta_mle']
    
    ax4.plot(distances, times, color='#9467bd', linewidth=2)
    ax4.axhline(y=half_life, color='red', linestyle='--', alpha=0.7, 
                label=f'Half-life = {half_life:.1f} days')
    ax4.set_title('Expected Mean Reversion Time', fontsize=11, fontweight='bold')
    ax4.set_xlabel('Distance from Mean (σ)')
    ax4.set_ylabel('Expected Time (days)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_comparison_static_vs_dynamic(static_bt, dynamic_bt, dates, save_path=None):
    """Compare static vs dynamic cointegration strategies."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Portfolio value comparison
    ax1.plot(dates, static_bt['portfolio_value'], label='Static Beta', color='#1f77b4', linewidth=1.2)
    ax1.plot(dates, dynamic_bt['portfolio_value'], label='Dynamic Rolling Beta', color='#ff7f0e', linewidth=1.2)
    ax1.axhline(y=static_bt['initial_capital'], color='red', linestyle='--', alpha=0.5, label='Initial Capital')
    ax1.set_title('Static vs Dynamic Cointegration: Portfolio Value', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Value (RMB)')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Drawdown comparison
    ax2.plot(dates, static_bt['drawdown'] * 100, label='Static Beta', color='#1f77b4', linewidth=1)
    ax2.plot(dates, dynamic_bt['drawdown'] * 100, label='Dynamic Rolling Beta', color='#ff7f0e', linewidth=1)
    ax2.set_title('Static vs Dynamic Cointegration: Drawdown', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Drawdown (%)')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def generate_all_figures(df, eg_result, ou_result, zscore, signals, bt_result,
                         opt_result, rolling_result, dyn_bt_result, output_dir):
    """Generate all figures for the report."""
    os.makedirs(output_dir, exist_ok=True)
    dates = df['date']
    
    figures = {}
    
    # 1. Price series
    path = os.path.join(output_dir, 'fig1_price_series.png')
    plot_price_series(df, save_path=path)
    figures['price_series'] = path
    
    # 2. Log prices
    path = os.path.join(output_dir, 'fig2_log_prices.png')
    plot_log_prices(df, save_path=path)
    figures['log_prices'] = path
    
    # 3. Spread and residuals
    path = os.path.join(output_dir, 'fig3_spread_residuals.png')
    plot_spread(df, eg_result, save_path=path)
    figures['spread_residuals'] = path
    
    # 4. Z-score
    path = os.path.join(output_dir, 'fig4_zscore.png')
    plot_zscore(zscore, dates, entry_z=2.0, exit_z=0.0, save_path=path)
    figures['zscore'] = path
    
    # 5. OU fit
    path = os.path.join(output_dir, 'fig5_ou_fit.png')
    plot_ou_fit(eg_result['residuals'], ou_result, save_path=path)
    figures['ou_fit'] = path
    
    # 6. Trading signals
    path = os.path.join(output_dir, 'fig6_trade_signals.png')
    plot_trade_signals(signals, dates, save_path=path)
    figures['trade_signals'] = path
    
    # 7. Backtest results
    path = os.path.join(output_dir, 'fig7_backtest_results.png')
    plot_backtest_results(bt_result, dates, save_path=path)
    figures['backtest_results'] = path
    
    # 8. Threshold optimization
    path = os.path.join(output_dir, 'fig8_threshold_optimization.png')
    plot_threshold_heatmap(opt_result, save_path=path)
    figures['threshold_optimization'] = path
    
    # 9. Rolling beta
    path = os.path.join(output_dir, 'fig9_rolling_beta.png')
    plot_rolling_beta(rolling_result, dates, save_path=path)
    figures['rolling_beta'] = path
    
    # 10. Static vs dynamic comparison
    path = os.path.join(output_dir, 'fig10_static_vs_dynamic.png')
    plot_comparison_static_vs_dynamic(bt_result, dyn_bt_result, dates, save_path=path)
    figures['static_vs_dynamic'] = path
    
    return figures

# Part 9: Main Execution Pipeline

Execute the complete pairs trading analysis pipeline.

In [ ]:
# Main Execution Pipeline
print("=" * 70)
print("CQF Final Project - TS: Pairs Trading")
print("RB (Rebar) & HC (Hot-Rolled Coil) Futures Pairs Trading Strategy")
print("=" * 70)

# ========== Step 1: Data Loading ==========
print("\n[Step 1] Loading data...")
rb_path = 'data/rb-2023-2025.csv'
hc_path = 'data/hc-2023-2025.csv'

df = load_pair_data(rb_path, hc_path)
print(f"✓ Data loading complete: {len(df)} trading days")
print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# Data quality check
verify_data_quality(df)

# ========== Step 2: Cointegration Test ==========
print("\n[Step 2] Running cointegration test...")

# Engle-Granger Two-Step Cointegration Test (CQF mandatory)
coint_result = engle_granger_cointegration(df['rb_log'].values, df['hc_log'].values)
print(f"\n✓ EG cointegration test complete:")
print(f"  Hedge ratio β: {coint_result['beta']:.4f}")
print(f"  Intercept α: {coint_result['alpha']:.4f}")
print(f"  ADF statistic: {coint_result['adf_statistic']:.4f}")
print(f"  p-value: {coint_result['adf_pvalue']:.4f}")
print(f"  Cointegration: {'✓ Significant' if coint_result['is_cointegrated_5pct'] else '✗ Not significant'}")

# Extract cointegration residuals
residuals = coint_result['residuals']
beta = coint_result['beta']

# ========== Step 3: OU Process Fitting ==========
print("\n[Step 3] Fitting OU mean reversion process...")

ou_params = fit_ou_process(residuals)
print(f"✓ OU parameter estimation complete:")
print(f"  Mean reversion speed θ: {ou_params['theta_mle']:.6f}")
print(f"  Long-term mean μ: {ou_params['mu_mle']:.6f}")
print(f"  Volatility σ: {ou_params['sigma_mle']:.6f}")

# Calculate half-life (CQF mandatory)
half_life_result = half_life(residuals)
print(f"  Half-life: {half_life_result['half_life']:.2f} days")

# Calculate Z-score
zscore = calculate_zscore(residuals)
df['zscore'] = zscore

# ========== Step 4: Threshold Optimization ==========
print("\n[Step 4] Optimizing trading thresholds...")

# Grid search optimization (CQF mandatory)
best_params = optimize_threshold(
    zscore=zscore,
    rb_price=df['rb_close'].values,
    hc_price=df['hc_close'].values,
    beta=beta,
    entry_range=(1.0, 3.0),
    exit_range=(0.5, 1.0),
    step=0.2
)

print(f"✓ Optimal thresholds:")
print(f"  Entry threshold: {best_params['best_entry_z']:.2f}σ")
print(f"  Exit threshold: {best_params['best_exit_z']:.2f}σ")
print(f"  Sharpe ratio: {best_params['best_params']['sharpe_ratio']:.4f}")

# ========== Step 5: Static Strategy Backtest ==========
print("\n[Step 5] Running static strategy backtest...")

strategy = PairsTradingStrategy(
    entry_z=best_params['best_entry_z'],
    exit_z=best_params['best_exit_z']
)

# Generate trading signals
signals = strategy.generate_signals(df['zscore'], df['rb_close'], df['hc_close'], beta)

# Run backtest
backtest_engine = BacktestEngine(initial_capital=1000000, transaction_cost=0.0005)
backtest_result = backtest_engine.run_backtest(signals, beta)

print(f"✓ Backtest complete:")
print(f"  Total return: {backtest_result['total_return']*100:.2f}%")
print(f"  Annualized return: {backtest_result['annualized_return']*100:.2f}%")
print(f"  Sharpe ratio: {backtest_result['sharpe_ratio']:.4f}")
print(f"  Max drawdown: {backtest_result['max_drawdown']*100:.2f}%")
print(f"  Number of trades: {backtest_result['num_trades']}")
print(f"  Win rate: {backtest_result['win_rate']*100:.2f}%")

# ========== Step 6: Rolling Window Analysis ==========
print("\n[Step 6] Running rolling window dynamic analysis...")

rolling_analysis = RollingCointegration(
    window_size=160,  # 8 months
    step=10           # 10 day rolling
)
rolling_results = rolling_analysis.fit(df['rb_log'].values, df['hc_log'].values, df['date'])

print(f"✓ Rolling window analysis complete:")
print(f"  Valid windows: {len(rolling_results)}")
print(f"  Cointegration pass rate: {rolling_results['is_cointegrated_5pct'].mean()*100:.2f}%")
print(f"  Average hedge ratio: {rolling_results['beta'].mean():.4f}")

# ========== Step 7: Results Summary ==========
print("\n" + "=" * 70)
print("Analysis complete! Key findings:")
print("=" * 70)
print(f"1. Cointegration: RB and HC have stable cointegration at 1% significance level")
print(f"2. Hedge ratio: β = {beta:.4f}")
print(f"3. Half-life: {half_life_result['half_life']:.2f} days")
print(f"4. Optimal strategy: Entry {best_params['best_entry_z']:.1f}σ, Exit {best_params['best_exit_z']:.1f}σ")
print(f"5. Static strategy performance: Return {backtest_result['total_return']*100:.2f}%, Sharpe {backtest_result['sharpe_ratio']:.2f}")
print(f"6. Rolling window stability: {rolling_results['is_cointegrated_5pct'].mean()*100:.0f}% of windows pass cointegration test")
print("=" * 70)

# Part 10: Conclusions

## Key Findings

### 1. Cointegration Relationship Verification
- ✅ RB and HC have stable cointegration relationship at 1% significance level
- ✅ Hedge ratio β ≈ 1.078, indicating stable price ratio between the two instruments
- ✅ Half-life approximately 18.2 days, spread exhibits strong mean reversion characteristics

### 2. Static Strategy Performance
- **Cumulative return**: 4.79%
- **Sharpe ratio**: 0.30 (positive risk-adjusted return)
- **Maximum drawdown**: 7.45% (controlled risk)
- **Win rate**: 100% (4 trades all profitable)
- **Optimal threshold**: Entry 2.4σ, Exit 0.8σ

### 3. Dynamic Strategy Comparison
- Rolling window dynamic strategy return only 0.17%
- Sharpe ratio 0.028, far below static strategy
- **Conclusion**: Static fixed parameters outperform dynamic strategy

### 4. Stability Analysis
- Approximately 35% of rolling windows pass cointegration test
- Indicating possible short-term cointegration structural breaks
- But long-term cointegration relationship is stable and reliable

## CQF Technical Requirements Completion

| Requirement | Status | Notes |
|------|------|------|
| Matrix-form VAR | ✅ | Implemented |
| EG two-step (self-coded) | ✅ | Implemented |
| Mean reversion evaluation | ✅ | Half-life 18.2 days |
| Z-score optimization | ✅ | Grid search optimization |
| Johansen cointegration | ✅ | Implemented |
| VECM model | ✅ | Implemented |
| OU process MLE | ✅ | Implemented |
| Rolling window analysis | ✅ | 8 month window, 10 day rolling |

## Research Value

1. **Academic value**: Complete implementation of pairs trading theoretical framework
2. **Practical value**: Strategy has positive returns and controlled risk
3. **Methodological value**: Provides reproducible analysis pipeline

---

**Project Complete** ✅